# Qwen3.5 → Mercury-2 diffusion LM **by the translator C** (architecture-agnostic, at scale)

**Method** (proven on Supra-50M, runs 1–7): `B* = C(donor)` — donor weights + per-matrix
SVD-frame corrections **emitted by C** (`Δy = ((x·V)·A(z)ᵀ)·Uᵀ`, input gauge-invariant /
output covariant) + an emitted [MASK] embedding. C trains through the diffusion loss on the
**self-zoo** (the donor's own depth-truncated sub-stacks); **all data is sampled from the
donor**; **B is never trained**.

**Qwen3.5 specifics** (researched after the first run): it is a **hybrid Gated-DeltaNet +
full-attention** model (3:1), possibly MoE — so the fixed q/k/v/o/gate/up/down assumption is
dropped: **C auto-discovers every `nn.Linear` per layer** and emits a delta for each from its
own signature (weight-tied over all matrices). Kaggle fixes: upgrade transformers≥5.8;
**upcast every DeltaNet Conv1D to fp32** (T4/SM7.5 has no fp16 conv engine).

**Honest ceiling:** DeltaNet layers are causal *by construction* (causal conv + recurrence);
only the full-attention layers go bidirectional via the mask. B* is therefore *partially*
bidirectional — a real architectural limit, reported as-is.

**Run:** accelerator **GPU T4 x2** + **Internet**. `PILOT=True` first (Qwen3.5-0.8B, ~20 min —
the smoke test at this scale), then `PILOT=False` for the 9B port. Gated repo → add `HF_TOKEN`.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import gc, json, math, time, torch
import torch.nn as nn
import torch.nn.functional as F

PILOT = True
MODEL_ID = 'Qwen/Qwen3.5-0.8B' if PILOT else 'Qwen/Qwen3.5-9B'
N_GEN    = 256 if PILOT else 512
N_HELD   = 32  if PILOT else 48
SEQ_LEN  = 160
C_STEPS  = 2000 if PILOT else 1500   # 9B step ~8-10s on 2xT4 -> 1500 keeps the session ~5-6h
C_BS     = 4   if PILOT else 2
LR       = 5e-4        # peak lr (warmup 100) -- higher than supra's 3e-4 to fit more in-budget
SUB      = 48          # SVD-frame correction subspace (A is SUB x SUB)
SIG_K    = 32          # singular values kept in a matrix's signature
D_Z      = 16
R_EMB    = 8           # learned per-role embedding (lets C distinguish matrix roles)
NROLE    = 40
MAX_PER_LAYER = 10     # cap nn.Linear per layer (largest by numel) -> bounds SVD cost on MoE
EPS_T    = 0.05
KD_LAMBDA, KD_TOPK = 0.3, 64
EVAL_EVERY = 200 if PILOT else 400
DEV0 = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('PILOT' if PILOT else '9B RUN', '| model', MODEL_ID, '| GPUs', torch.cuda.device_count())

In [ ]:
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError
def _tf_ver():
    try: return tuple(int(x) for x in version('transformers').split('.')[:2])
    except PackageNotFoundError: return (0, 0)
if _tf_ver() < (5, 8):
    # ignore pip's return code: it exits non-zero on Kaggle's PRE-EXISTING dask/cuml/numba
    # conflicts even when transformers installs fine. Judge success by the on-disk version,
    # read via importlib.metadata (NOT by importing transformers, which would cache 5.0.0).
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'transformers>=5.8', 'accelerate'])
if _tf_ver() < (5, 8):
    raise RuntimeError('Need transformers>=5.8 for model_type qwen3_5 but it is not installed. '
                       'Enable Internet (right panel -> Session options -> Internet -> ON) and Run All.')
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers; print('transformers', transformers.__version__)
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True,
    device_map='balanced' if torch.cuda.device_count() > 1 else DEV0, attn_implementation='eager')
model.eval()
for p in model.parameters(): p.requires_grad_(False)

# T4 fp16 Conv1D (DeltaNet causal conv) has no cuDNN engine -> compute conv in fp32.
# Patch at the FUNCTIONAL level (not module.forward) so accelerate's device hooks are untouched.
_orig_conv1d = F.conv1d
def _conv1d_fp32(inp, weight, bias=None, *a, **k):
    if inp.dtype == torch.float16:
        o = _orig_conv1d(inp.float(), weight.float(), None if bias is None else bias.float(), *a, **k)
        return o.to(inp.dtype)
    return _orig_conv1d(inp, weight, bias, *a, **k)
F.conv1d = _conv1d_fp32; torch.nn.functional.conv1d = _conv1d_fp32
print('conv1d fp32 shim installed')

core = model.model; layers = core.layers; L_FULL = len(layers)
D = model.config.hidden_size; V = model.config.vocab_size; MASK_ID = V
E_W = core.embed_tokens.weight
lt = getattr(model.config, 'layer_types', None)
print(f'{MODEL_ID}: L={L_FULL} d={D} V={V} tied={model.config.tie_word_embeddings}')
print('layer_types:', lt if lt else '(uniform)')
# architecture report: nn.Linear inventory of an early and a late layer
for li in (0, L_FULL // 2):
    inv = [(n, tuple(mm.weight.shape)) for n, mm in layers[li].named_modules() if isinstance(mm, nn.Linear)]
    print(f'  layer {li}: {len(inv)} Linear; sample {[n for n,_ in inv][:8]}')
TRUNC_DEPTHS = sorted({max(2, L_FULL // 5), 2 * L_FULL // 5, 3 * L_FULL // 5, 4 * L_FULL // 5, L_FULL - 1})
print('self-zoo truncation depths:', TRUNC_DEPTHS, '| full', L_FULL, '(held out)')

In [ ]:
def make_bias(B, T, causal, device, dtype=torch.float16):
    m = (torch.full((T, T), torch.finfo(dtype).min, device=device, dtype=dtype).triu(1)
         if causal else torch.zeros((T, T), device=device, dtype=dtype))
    return m[None, None].expand(B, 1, T, T)

def run_stack(ids, Lt, causal=False, mask_row=None):
    """First Lt layers + final norm + lm_head, with an explicit 4D mask (bidirectional on the
    full-attention layers; DeltaNet layers stay causal by construction) and [MASK]-embedding
    injection. Differentiable wrt whatever the hooks add (the donor is frozen)."""
    B, T = ids.shape
    h = core.embed_tokens(ids.to(E_W.device).clamp_max(V - 1))
    if mask_row is not None:
        h = torch.where((ids.to(E_W.device) == MASK_ID)[..., None], mask_row.to(h.dtype), h)
    pos_ids = torch.arange(T, device=h.device)[None].expand(B, T)
    pe = core.rotary_emb(h, pos_ids) if hasattr(core, 'rotary_emb') else None
    bias = make_bias(B, T, causal, h.device)
    for l in range(Lt):
        lay = layers[l]; dev = lay.input_layernorm.weight.device
        kw = dict(attention_mask=bias.to(dev), position_ids=pos_ids.to(dev))
        if pe is not None: kw['position_embeddings'] = tuple(p.to(dev) for p in pe)
        try: out = lay(h, **kw)
        except TypeError:
            kw.pop('position_embeddings', None); out = lay(h, **kw)
        h = out[0] if isinstance(out, tuple) else out
    h = core.norm(h.to(core.norm.weight.device))
    return model.lm_head(h.to(model.lm_head.weight.device))

with torch.no_grad():
    p1 = torch.randint(4, 1000, (1, 8), device=DEV0)
    a = run_stack(p1, L_FULL, causal=True)[0, 0]
    p2 = p1.clone(); p2[0, -1] = 7
    b = run_stack(p2, L_FULL, causal=True)[0, 0]
    c = run_stack(p2, L_FULL, causal=False)[0, 0]
    assert torch.allclose(a, b, atol=1e-2), 'causal leak!'
    print('run_stack ok | bidirectional changes output:', not torch.allclose(a, c, atol=1e-2),
          '(only full-attn layers go bidirectional; DeltaNet stays causal)')

In [ ]:
@torch.no_grad()
def gen_batch(prompts, n_new, temperature):
    am = (prompts != (tok.pad_token_id or 0)).long()
    out = model.generate(prompts, attention_mask=am, max_new_tokens=n_new, do_sample=True,
                         temperature=temperature, top_k=50, pad_token_id=tok.pad_token_id or 0)
    return out[:, prompts.shape[1]:]

t0 = time.time()
bos_id = tok.bos_token_id if tok.bos_token_id is not None else (tok.pad_token_id or 0)
boot = gen_batch(torch.full((24, 8), bos_id, dtype=torch.long, device=DEV0), SEQ_LEN // 2, 1.0)
uni = torch.bincount(boot.reshape(-1).cpu().clamp_max(V - 1), minlength=V).float() + 1e-3
uni = uni / uni.sum()
need, gb = N_GEN + N_HELD, 16
chunks, temps = [], [0.7, 0.9, 1.0]
while sum(c.shape[0] for c in chunks) < need:
    tt = temps[len(chunks) % len(temps)]
    pr = torch.multinomial(uni.expand(gb, V), 4, replacement=True).to(DEV0)
    chunks.append(gen_batch(pr, SEQ_LEN, tt).cpu())
corpus = torch.cat(chunks, 0)[:need].clamp_max(V - 1)
train_ids, held_ids = corpus[:N_GEN].to(DEV0), corpus[N_GEN:].to(DEV0)
print(f'corpus from donor: train {tuple(train_ids.shape)} held {tuple(held_ids.shape)} in {time.time()-t0:.0f}s')
print('sample:', repr(tok.decode(train_ids[0, :48])))
gc.collect(); torch.cuda.empty_cache()

In [ ]:
def sample_mask_rate(b, device=DEV0): return EPS_T + (1.0 - EPS_T) * torch.rand(b, device=device)
def forward_mask(x0, t):
    B, L = x0.shape; noise = torch.rand(B, L, device=x0.device); m = noise < t[:, None].expand(B, L)
    empty = ~m.any(dim=1)
    if empty.any(): m[empty.nonzero(as_tuple=True)[0], noise[empty].argmin(dim=1)] = True
    return torch.where(m, torch.full_like(x0, MASK_ID), x0), m
def masked_diffusion_loss(logits, x0, m, t, kd_probs=None, kd_idx=None):
    B, T = x0.shape; lm = logits[m].float()
    ce = F.cross_entropy(lm, x0[m], reduction='none')
    if kd_probs is not None:
        q = lm.log_softmax(-1); ce = (1 - KD_LAMBDA) * ce - KD_LAMBDA * (kd_probs * q.gather(1, kd_idx)).sum(1)
    seq_idx = torch.arange(B, device=x0.device)[:, None].expand(B, T)[m]
    return (torch.zeros(B, device=ce.device).index_add_(0, seq_idx, ce) / (t * T)).mean()
@torch.no_grad()
def masked_ce_at(fn, ids, t_val, n=16):
    x0 = ids[:n]; x_t, m = forward_mask(x0, torch.full((x0.shape[0],), t_val, device=x0.device))
    return F.cross_entropy(fn(x_t)[m].float(), x0[m]).item()
@torch.no_grad()
def denoise_v2(fn, ids, frozen, steps=64, temp0=1.0, alg_temp=0.3, remask_frac=0.15, prior=None,
               beta=0.6, samp_beta=0.4, rep_gamma=0.8, ban_ids=None, temp_floor=0.7, no_repeat=3):
    B, L = ids.shape; n0 = int(((ids == MASK_ID) & ~frozen).sum().item())
    if n0 == 0: return ids
    lp = None if prior is None else torch.log(prior.clamp_min(1e-8)).to(ids.device)
    for s in range(steps):
        masked = (ids == MASK_ID) & ~frozen; n_left = int(masked.sum().item())
        if n_left == 0: break
        logits = fn(ids).float()
        if ban_ids is not None:
            for b in ban_ids: logits[..., b] = float('-inf')
        if lp is not None and samp_beta > 0: logits = logits - samp_beta * lp
        if rep_gamma > 0:
            for b in range(B):
                seen = ids[b][ids[b] != MASK_ID]
                if seen.numel(): logits[b] = logits[b] - rep_gamma * torch.log1p(torch.bincount(seen, minlength=V).float())
        if no_repeat > 0:
            for b in range(B):
                row = ids[b].tolist(); seen = {}
                for j in range(L - no_repeat + 1):
                    sg = row[j:j + no_repeat]
                    if MASK_ID in sg: continue
                    seen.setdefault(tuple(sg[:-1]), set()).add(sg[-1])
                for i in range(no_repeat - 1, L):
                    ctx = tuple(row[i - (no_repeat - 1):i])
                    if MASK_ID in ctx: continue
                    for tk in seen.get(ctx, ()): logits[b, i, tk] = float('-inf')
        tau = temp0 * max(0.0, 1.0 - s / max(1, steps - 1))
        if s < int(0.9 * steps): tau = max(tau, temp_floor)
        if tau > 0.05:
            probs = (logits / tau).softmax(-1); pred = torch.multinomial(probs.view(-1, V), 1).view(B, L)
        else:
            probs = logits.softmax(-1); pred = probs.argmax(-1)
        conf = probs.gather(-1, pred[..., None]).squeeze(-1).clamp_min(1e-9).log()
        if lp is not None: conf = conf - beta * lp[pred]
        conf = conf.masked_fill(~masked, float('-inf'))
        if alg_temp > 0:
            g = torch.rand_like(conf).clamp_min(1e-9); conf = conf + alg_temp * (-(-g.log()).log()) * (s < steps // 2)
        k = max(1, min(n_left, n_left - int(n0 * math.cos(math.pi / 2 * (s + 1) / steps))))
        ids.view(-1)[conf.view(-1).topk(k).indices] = pred.view(-1)[conf.view(-1).topk(k).indices]
        if remask_frac > 0 and s < steps // 2:
            rev = (ids != MASK_ID) & ~frozen
            if rev.any():
                cur = probs.gather(-1, ids.clamp_max(V - 1)[..., None]).squeeze(-1).masked_fill(~rev, float('inf'))
                q = max(1, int(rev.sum().item() * remask_frac)); ids.view(-1)[(-cur.view(-1)).topk(q).indices] = MASK_ID
    masked = (ids == MASK_ID) & ~frozen
    if masked.any(): ids = torch.where(masked, fn(ids).float().argmax(-1), ids)
    return ids
@torch.no_grad()
def semi_ar_generate(fn, prompt, total_len, block=16, steps_per_block=16, **kw):
    ids = prompt.clone()
    while ids.shape[1] < total_len:
        b = min(block, total_len - ids.shape[1])
        win = torch.cat([ids, torch.full((ids.shape[0], b), MASK_ID, dtype=torch.long, device=ids.device)], 1)
        fr = torch.zeros_like(win, dtype=torch.bool); fr[:, :ids.shape[1]] = True
        ids = denoise_v2(fn, win, fr, steps=steps_per_block, **kw)
    return ids
print('diffusion core + samplers ready')

In [ ]:
# Discover every nn.Linear in each decoder layer (DeltaNet in/out proj, full-attn q/k/v/o,
# MoE expert/router linears, MLP) -> cap to the MAX_PER_LAYER largest by numel per layer.
KNOWN_ROLES = ['out_proj','in_proj_qkv','in_proj_z','in_proj_b','in_proj_a','gate_proj',
               'up_proj','down_proj','q_proj','k_proj','v_proj','o_proj','gate']
def role_id(leaf):
    return KNOWN_ROLES.index(leaf) if leaf in KNOWN_ROLES else \
           len(KNOWN_ROLES) + (sum(map(ord, leaf)) % (NROLE - len(KNOWN_ROLES)))
LTYPE = getattr(model.config, 'layer_types', None)
def is_linear(l): return bool(LTYPE) and LTYPE[l] == 'linear_attention'

t0 = time.time(); LINS = []
with torch.no_grad():
    for l in range(L_FULL):
        cand = [(n, mm) for n, mm in layers[l].named_modules()
                if isinstance(mm, nn.Linear) and mm.weight.ndim == 2 and min(mm.weight.shape) >= 16]
        cand.sort(key=lambda nm: -nm[1].weight.numel())
        for name, mm in cand[:MAX_PER_LAYER]:
            W = mm.weight.float()
            U, S, Vv = torch.svd_lowrank(W, q=min(SUB + 8, min(W.shape) - 1), niter=4)
            r = min(SUB, U.shape[1])
            mm._Ud = U[:, :r].to(mm.weight.dtype).contiguous()
            mm._Vd = Vv[:, :r].to(mm.weight.dtype).contiguous()
            mm._r = r; mm._A = None; mm._on = False; mm._layer = l
            mm._role = role_id(name.split('.')[-1])
            spec = torch.zeros(SIG_K); s = (S / (S.norm() + 1e-9))[:SIG_K]; spec[:s.numel()] = s.cpu()
            mm._sig = torch.cat([spec, torch.tensor([math.log(W.norm().item() + 1e-9) / 10,
                math.log(W.shape[0]) / 12, math.log(W.shape[1]) / 12, l / (L_FULL - 1),
                1.0 if is_linear(l) else 0.0])]).to(DEV0)   # +depth +type(linear/full)
            LINS.append(mm)
        if torch.cuda.is_available() and l % 8 == 0: torch.cuda.empty_cache()
SIG_DIM = SIG_K + 5
print(f'{len(LINS)} Linear matrices translated ({time.time()-t0:.0f}s); distinct roles:',
      len({mm._role for mm in LINS}))

def _delta_hook(mod, inp, out):
    if getattr(mod, '_on', False) and mod._A is not None:
        return out + ((inp[0] @ mod._Vd) @ mod._A.T) @ mod._Ud.T
    return out
for mm in LINS: mm.register_forward_hook(_delta_hook)

PRESENT_ROLES = sorted({mm._role for mm in LINS})
ROLE_SLOT = {r: i for i, r in enumerate(PRESENT_ROLES)}

class TranslatorC(nn.Module):
    """v4 (math-grounded): PER-ROLE output generators M_role -- with one shared M all
    corrections live in a single d_z-dim subspace of R^{sub^2}; per-role generators give
    each role its own subspace (this is the structure that produced the decisive control
    separation on supra). Input: signature + learned role embedding (v2 signatures identify
    the role at only ~33-39% LOO accuracy -- spectra of trained matrices are near-universal,
    within-role cos 0.9954 vs cross-role 0.9900 -- so explicit identity is required)."""
    def __init__(self, d_z=D_Z, sub=SUB, h=160):
        super().__init__(); self.sub = sub
        self.role_emb = nn.Embedding(NROLE, R_EMB)
        self.enc = nn.Sequential(nn.Linear(SIG_DIM + R_EMB, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, d_z))
        self.M = nn.ParameterDict({str(r): nn.Parameter(torch.zeros(sub * sub, d_z))
                                   for r in PRESENT_ROLES})    # zero-init => B == floor
        self.mask_logits = nn.Parameter(torch.zeros(V))
    def mask_row(self): return self.mask_logits.softmax(0).to(E_W.device, E_W.dtype) @ E_W
    def _z(self, mm):
        re = self.role_emb(torch.tensor(mm._role, device=mm._sig.device))
        return self.enc(torch.cat([mm._sig, re]))
    def _A(self, mm):
        return (self.M[str(mm._role)] @ self._z(mm)).view(self.sub, self.sub)
    def install(self, Lt, deltas=True):
        for mm in LINS:
            if not deltas or mm._layer >= Lt: mm._on = False; continue
            A = self._A(mm)
            mm._A = A[:mm._r, :mm._r].to(mm.weight.device, mm.weight.dtype); mm._on = True
    def uninstall(self):
        for mm in LINS: mm._on = False
C = TranslatorC().to(DEV0)
print(f'C params: {sum(p.numel() for p in C.parameters())} | per-role generators: {len(PRESENT_ROLES)}')

In [ ]:
opt = torch.optim.AdamW(C.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 100))
ema, t0 = None, time.time()
for step in range(1, C_STEPS + 1):
    Lt = TRUNC_DEPTHS[step % len(TRUNC_DEPTHS)]
    x0 = train_ids[torch.randint(0, train_ids.shape[0], (C_BS,), device=DEV0)]
    t = sample_mask_rate(C_BS); x_t, m = forward_mask(x0, t)
    kd_probs = kd_idx = None
    if KD_LAMBDA > 0:
        C.uninstall()
        with torch.no_grad():
            tl = run_stack(x0, L_FULL, causal=True)
            tl = torch.cat([tl[:, :1] * 0, tl[:, :-1]], 1)
            kd_probs, kd_idx = tl[m].float().softmax(-1).topk(KD_TOPK, dim=-1)
            kd_probs = kd_probs / kd_probs.sum(-1, keepdim=True); del tl
    C.install(Lt)
    logits = run_stack(x_t, Lt, causal=False, mask_row=C.mask_row())
    loss = masked_diffusion_loss(logits, x0, m, t, kd_probs, kd_idx)
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(C.parameters(), 1.0)
    opt.step(); sched.step(); C.uninstall()
    ema = loss.item() if ema is None else 0.98 * ema + 0.02 * loss.item()
    if step % EVAL_EVERY == 0 or step == 1:
        with torch.no_grad():
            Lp = TRUNC_DEPTHS[1]; C.install(Lp)
            hce = masked_ce_at(lambda i: run_stack(i, Lp, causal=False, mask_row=C.mask_row()), held_ids, 0.5, n=8)
            C.uninstall()
        print(f'step {step:>5}  train(ema) {ema:6.3f}  sub-stack-L{Lp} masked-CE@0.5 {hce:5.2f}  ({time.time()-t0:.0f}s)')
print('C trained — the donor was never trained, only C was')

In [ ]:
with torch.no_grad(): mrow = C.mask_row(); mean_row = E_W.float().mean(0).to(E_W.dtype)
def fnB(i): return run_stack(i, L_FULL, causal=False, mask_row=mrow)
def fn0(i): return run_stack(i, L_FULL, causal=False, mask_row=mean_row)

@torch.no_grad()
def masked_ce_paired(fn, ids, t_val, n=16, seed=777):
    """PAIRED evaluation: a fixed mask draw shared by every condition (cuts the variance of
    between-condition differences -- the F3 statistical fix)."""
    g = torch.Generator(device='cpu').manual_seed(seed)
    x0 = ids[:n]
    noise = torch.rand(x0.shape, generator=g).to(x0.device)
    m = noise < t_val
    if (~m.any(dim=1)).any(): m[:, 0] = True
    x_t = torch.where(m, torch.full_like(x0, MASK_ID), x0)
    return F.cross_entropy(fn(x_t)[m].float(), x0[m]).item()

with torch.no_grad():
    al = run_stack(held_ids[:8, :-1], L_FULL, causal=True)
    ar = F.cross_entropy(al.reshape(-1, V).float(), held_ids[:8, 1:].reshape(-1)).item(); del al
print(f'uniform ln(V) = {math.log(V):.2f} | donor AR next-token CE (held) = {ar:.2f}')

# ---- F2: ATTRIBUTION -- decompose the floor->B* gain (paired masks) ----
print('\nheld masked-CE (paired masks):  floor | mask-row-only | deltas-only | full B*')
for tv in (0.3, 0.5, 0.7, 0.9):
    C.uninstall()
    a_fl = masked_ce_paired(fn0, held_ids, tv)                         # nothing translated
    a_mr = masked_ce_paired(fnB, held_ids, tv)                         # mask row only
    C.install(L_FULL)
    a_do = masked_ce_paired(fn0, held_ids, tv)                         # deltas only
    a_bb = masked_ce_paired(fnB, held_ids, tv)                         # full B*
    C.uninstall()
    print(f'  t={tv}:   {a_fl:6.2f} | {a_mr:6.2f} | {a_do:6.2f} | {a_bb:6.2f}')

x0 = held_ids[:4]; x_c, m = forward_mask(x0, torch.full((x0.shape[0],), 0.25, device=x0.device))
for name, fn, ins in (('floor', fn0, False), ('B*', fnB, True)):
    if ins: C.install(L_FULL)
    rec = denoise_v2(fn, x_c.clone(), ~m, steps=20, temp0=0.0, alg_temp=0.0, remask_frac=0.0, ban_ids=None, temp_floor=0.0, no_repeat=0)
    if ins: C.uninstall()
    acc = ((rec == x0) & m).sum().item() / m.sum().item()
    print(f'\nreconstruction ({name}): token accuracy {acc:.1%}')
    if ins:
        print('  original :', repr(tok.decode(x0[0, :48])))
        print('  recovered:', repr(tok.decode(rec[0, :48].clamp_max(V - 1))))

UNI = uni.to(DEV0)
kw = dict(temp0=1.0, alg_temp=0.3, remask_frac=0.15, prior=UNI, beta=0.6, samp_beta=0.4, rep_gamma=0.8, temp_floor=0.7, no_repeat=3)
uniq = torch.tensor([held_ids[i].unique().numel() for i in range(held_ids.shape[0])])
prompt = held_ids[uniq.argsort(descending=True)[:2], :32]
C.install(L_FULL)
g4 = semi_ar_generate(fnB, prompt, 32 + 64, block=16, steps_per_block=16, **kw)
C.uninstall()
for b in range(2):
    print(f'\nprompt {b}   :', repr(tok.decode(prompt[b])))
    print('semi-AR cont:', repr(tok.decode(g4[b, 32:].clamp_max(V - 1))))

# ---- F3: statistically sound controls -- paired masks, k=5 draws, TWO granularities ----
o_sig = [mm._sig for mm in LINS]; o_role = [mm._role for mm in LINS]
def restore():
    for i, mm in enumerate(LINS): mm._sig, mm._role = o_sig[i], o_role[i]
def ctrl_eval():
    C.install(L_FULL); v = masked_ce_paired(fnB, held_ids, 0.5); C.uninstall(); return v

C.install(L_FULL); bb = masked_ce_paired(fnB, held_ids, 0.5); C.uninstall()
res = {}
for mode in ('cross-role', 'within-role'):
    vals = []
    for k in range(5):
        g = torch.Generator().manual_seed(100 + k)
        if mode == 'cross-role':                       # destroy role AND signature identity
            perm = torch.randperm(len(LINS), generator=g).tolist()
            for i, mm in enumerate(LINS): mm._sig, mm._role = o_sig[perm[i]], o_role[perm[i]]
        else:                                          # permute only WITHIN each role class
            for r in set(o_role):
                idxs = [i for i in range(len(LINS)) if o_role[i] == r]
                pr = torch.randperm(len(idxs), generator=g).tolist()
                for a, i in enumerate(idxs): LINS[i]._sig = o_sig[idxs[pr[a]]]
        vals.append(ctrl_eval()); restore()
    v = torch.tensor(vals); res[mode] = v
    print(f'control {mode:>11}-shuffle masked-CE@0.5: {v.mean():.2f} ± {v.std():.2f}  '
          f'(B* {bb:.2f}; Δ={v.mean()-bb:+.2f})')
print('  reading: cross-role >> B* => C uses ROLE identity; within-role > B* => C also uses',
      'finer-than-role (spectra/depth) information')

# probe: deltas on layers seen in truncation training only (0..L-2) vs all
C.install(L_FULL - 1); p1 = masked_ce_paired(fnB, held_ids, 0.5); C.uninstall()
print(f'probe — deltas on 0..{L_FULL-2} only: {p1:.2f} (vs all {bb:.2f})')

In [ ]:
from safetensors.torch import save_file
pack = {'mercury.mask_embedding': mrow.detach().float().cpu()}
with torch.no_grad():
    for i, mm in enumerate(LINS):
        A = C._A(mm)[:mm._r, :mm._r]
        pack[f'lin{i}.UA'] = (mm._Ud.float().cpu() @ A.cpu())
        pack[f'lin{i}.V'] = mm._Vd.float().cpu()
path = ('/kaggle/working/' if os.path.isdir('/kaggle/working') else '') + \
       ('qwen35_pilot_pack.safetensors' if PILOT else 'qwen35_9b_mercury_pack.safetensors')
save_file(pack, path); torch.save(C.state_dict(), path.replace('.safetensors', '_C.pt'))
print('saved translation pack ->', path, f'({os.path.getsize(path)/1e6:.0f} MB), {len(LINS)} matrices')

## How to read
- **Architecture report** (cell 3): confirms layer_types (DeltaNet vs full-attn) and the
  per-layer Linear inventory C adapts to. **PILOT is the smoke test** — run it first.
- **floor vs B\*** masked-CE, **reconstruction**, the **shuffled-signature control** (must
  separate ⇒ C reads the weights), and semi-AR generation — same evidence structure as supra.
- **Architectural ceiling:** only full-attention layers go bidirectional (DeltaNet is causal
  by construction), so a hybrid donor is intrinsically harder to turn into a denoiser than a
  pure transformer — read B\* vs floor as the translation signal, not against a dense ceiling.
- **B is never trained; its weights are never modified** (deltas live in forward hooks; the
  deliverable is a translation pack applying B\* anywhere).